In [16]:
# D&D 5e Spell RAG System - Clean Implementation
# Cell 1: Setup and Imports

import os
import re
import json
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Set, Tuple
from collections import defaultdict

# Vector database and embeddings
import chromadb
from chromadb.config import Settings

# For token estimation
import tiktoken

print("🎲 D&D 5e Spell RAG System")
print("📚 Loading dependencies...")
print("✅ All imports loaded successfully!")

🎲 D&D 5e Spell RAG System
📚 Loading dependencies...
✅ All imports loaded successfully!


In [17]:
# Cell 2: Core Data Structures

@dataclass
class SpellMetadata:
    """Complete spell metadata for D&D 5e spells"""
    name: str
    level: int
    school: str
    casting_time: str
    range: str
    components: str
    duration: str
    classes: List[str] = field(default_factory=list)

    # Optional enhanced metadata
    damage_type: Optional[str] = None
    save_type: Optional[str] = None
    attack_type: Optional[str] = None
    ritual: bool = False
    concentration: bool = False

    def to_dict(self) -> dict:
        """Convert to dictionary for ChromaDB metadata"""
        return {
            "spell_name": self.name,
            "level": self.level,
            "school": self.school,
            "classes": ",".join(self.classes),
            "casting_time": self.casting_time,
            "range": self.range,
            "components": self.components,
            "duration": self.duration,
            "ritual": self.ritual,
            "concentration": self.concentration
        }

@dataclass
class SpellChunk:
    """A chunk of spell content for RAG retrieval"""
    spell_name: str
    content: str
    chunk_type: str  # 'full', 'mechanics', 'description'
    metadata: SpellMetadata
    tags: Set[str] = field(default_factory=set)

    def get_retrieval_text(self) -> str:
        """Get formatted text for embedding"""
        return f"**{self.spell_name}** ({self.chunk_type})\n{self.content}"

print("✅ Data classes defined!")
print("   📋 SpellMetadata: Complete spell information")
print("   📦 SpellChunk: RAG-ready content chunks")

✅ Data classes defined!
   📋 SpellMetadata: Complete spell information
   📦 SpellChunk: RAG-ready content chunks


In [18]:
# Cell 3: Spell List Processor

class SpellListProcessor:
    """Extract class and level information from all_spells_copy.txt"""

    def __init__(self):
        self.class_spell_mapping = {}  # spell_name -> [classes]
        self.spell_level_mapping = {}  # spell_name -> level

    def process_spell_lists(self, all_spells_text: str) -> None:
        """Process the markdown format spell lists correctly"""
        print("🔍 Processing spell class and level data...")

        lines = all_spells_text.split('\n')
        current_class = None
        current_level = None
        total_mappings = 0

        for line in lines:
            line = line.strip()
            if not line:
                continue

            # Check for class header (## BARD SPELLS)
            class_match = re.match(r'## ([A-Z]+) SPELLS', line)
            if class_match:
                current_class = class_match.group(1).title()
                print(f"   📖 Processing {current_class} spells...")
                continue

            # Check for level header (### 1st Level: or ### Cantrips)
            level_match = re.match(r'### (.*?):', line)
            if level_match:
                level_text = level_match.group(1)
                if 'Cantrips' in level_text or '0 Level' in level_text:
                    current_level = 0
                else:
                    # Extract number from "1st Level", "2nd Level", etc.
                    level_num_match = re.search(r'(\d+)(?:st|nd|rd|th)', level_text)
                    if level_num_match:
                        current_level = int(level_num_match.group(1))
                    else:
                        current_level = None
                continue

            # Process spell lines - look for comma-separated spell names
            if current_class and current_level is not None and ',' in line:
                # This line contains spell names
                spell_names = [name.strip() for name in line.split(',')]

                for spell_name in spell_names:
                    spell_name = self._clean_spell_name(spell_name)
                    if spell_name:
                        # Add to class mapping
                        if spell_name not in self.class_spell_mapping:
                            self.class_spell_mapping[spell_name] = []
                        if current_class not in self.class_spell_mapping[spell_name]:
                            self.class_spell_mapping[spell_name].append(current_class)

                        # Add level mapping
                        self.spell_level_mapping[spell_name] = current_level
                        total_mappings += 1

        print(f"✅ Processed {total_mappings} spell-class mappings")
        print(f"📊 Found {len(self.spell_level_mapping)} unique spells")
        print(f"📊 Found {len(self.class_spell_mapping)} spells with class info")

    def _clean_spell_name(self, name: str) -> str:
        """Clean and normalize spell name"""
        if not name or len(name) < 3:
            return ""

        # Remove common artifacts
        name = name.strip()
        name = re.sub(r'\s+', ' ', name)  # Collapse whitespace

        # Skip numbers, counts, and obvious non-spell text
        if re.match(r'^\d+', name) or 'spells' in name.lower():
            return ""

        # Skip summary text and headers
        skip_phrases = ['total', 'level', 'cantrips', 'grand total', 'source:', 'chapter']
        if any(phrase in name.lower() for phrase in skip_phrases):
            return ""

        return name.title()

    def get_spell_info(self, spell_name: str) -> tuple:
        """Get level and classes for a spell"""
        level = self.spell_level_mapping.get(spell_name, -1)
        classes = self.class_spell_mapping.get(spell_name, [])
        return level, classes

print("✅ SpellListProcessor ready!")
print("   📋 Extracts class and level data from all_spells_copy.txt")
print("   🎯 Handles markdown format with ### headers")

✅ SpellListProcessor ready!
   📋 Extracts class and level data from all_spells_copy.txt
   🎯 Handles markdown format with ### headers


In [19]:
# Cell 4: Spell Text Parser for spells.txt

class SpellTextParser:
    """Parse spells.txt with UPPERCASE spell names and clean formatting"""

    def __init__(self):
        self.encoding = tiktoken.get_encoding("cl100k_base")

    def parse_spells_file(self, file_path: str) -> List[Tuple[SpellMetadata, str]]:
        """Parse the spells.txt file and return spell metadata + descriptions"""
        print(f"📖 Reading spells from: {file_path}")

        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
        except FileNotFoundError:
            raise FileNotFoundError(f"Spells file not found: {file_path}")

        print(f"📄 Loaded {len(content):,} characters")

        # Split into individual spell blocks
        spell_blocks = self._split_into_spell_blocks(content)
        print(f"🔍 Found {len(spell_blocks)} spell blocks")

        # Parse each spell block
        parsed_spells = []
        failed_parses = 0

        for i, block in enumerate(spell_blocks):
            try:
                metadata, description = self._parse_spell_block(block)
                parsed_spells.append((metadata, description))

                if (i + 1) % 50 == 0:
                    print(f"   ⚙️ Parsed {i + 1}/{len(spell_blocks)} spells...")

            except Exception as e:
                failed_parses += 1
                if failed_parses <= 5:  # Show first few errors
                    print(f"   ⚠️ Failed to parse spell {i+1}: {e}")

        print(f"✅ Successfully parsed {len(parsed_spells)} spells")
        if failed_parses > 0:
            print(f"⚠️ Failed to parse {failed_parses} spells")

        return parsed_spells

    def _split_into_spell_blocks(self, content: str) -> List[str]:
        """Split content into individual spell blocks - handles OCR errors"""
        spell_pattern = r'\n(?=[A-Z][A-Z\s\']{2,}\s*\n)'

        blocks = re.split(spell_pattern, content)

        # Filter blocks that actually contain spell structure
        valid_blocks = []
        for block in blocks:
            block = block.strip()
            if (len(block) > 50 and self._has_spell_structure(block)):
                valid_blocks.append(block)

        return valid_blocks

    def _normalize_ocr(self, text: str) -> str:
        """Normalize common OCR errors"""
        return (text.replace("levei", "level")
                    .replace("leve1", "level")
                    .replace("leve!", "level")
                    .replace("leveı", "level")   # dotless i
                    .replace("leveI", "level")
                    .replace("casling", "casting")
                    .replace("aclion", "action")
                    .replace("feel", "feet")
                    .replace("crealure", "creature")
                    .replace("crealures", "creatures")
                    .replace("wilhin", "within")
                    .replace("whiIe", "while")
                    .replace("poi nt", "point")
                    .replace("poinl", "point")
                    .replace("Id", "1d")  # OCR I vs 1
                    )

    def _has_spell_structure(self, block: str) -> bool:
        """Check if block has spell structure - very lenient for OCR errors"""
        block_lower = self._normalize_ocr(block.lower())

        level_indicators = [
            'cantrip', 'level',
            'abjuration', 'conjuration', 'divination', 'enchantment',
            'evocation', 'illusion', 'necromancy', 'transmutation'
        ]

        casting_indicators = [
            'casting time', 'casting', 'action'
        ]

        component_indicators = [
            'components', 'component', 'v, s', 'v,s'
        ]

        has_level_info = any(indicator in block_lower for indicator in level_indicators)
        has_casting_info = any(indicator in block_lower for indicator in casting_indicators)
        has_components = any(indicator in block_lower for indicator in component_indicators)

        return has_level_info and (has_casting_info or has_components)

    def _parse_spell_block(self, block: str) -> Tuple[SpellMetadata, str]:
        """Parse a single spell block into metadata and description"""
        block = self._normalize_ocr(block)
        lines = [line.strip() for line in block.split('\n') if line.strip()]

        if len(lines) < 6:
            raise ValueError("Spell block too short")

        # First line is spell name (UPPERCASE)
        spell_name = lines[0].strip().title()

        # Second line is level and school
        level_school_line = lines[1]
        level, school, ritual = self._parse_level_school(level_school_line)

        # Extract standard fields
        casting_time = self._extract_field(block, "Casting Time")
        range_val = self._extract_field(block, "Range")
        components = self._extract_field(block, "Components")
        duration = self._extract_field(block, "Duration")

        # Check for concentration
        concentration = 'concentration' in duration.lower()

        # Find description (after the Duration line)
        description = self._extract_description(block)

        # Create metadata
        metadata = SpellMetadata(
            name=spell_name,
            level=level,
            school=school,
            casting_time=casting_time,
            range=range_val,
            components=components,
            duration=duration,
            ritual=ritual,
            concentration=concentration
        )

        return metadata, description

    def _parse_level_school(self, line: str) -> Tuple[int, str, bool]:
        """Parse level and school from lines like '3rd-level evocation' or 'Cantrip evocation'"""
        line = self._normalize_ocr(line.lower().strip())
        ritual = 'ritual' in line

        # Extract level
        if 'cantrip' in line:
            level = 0
        else:
            level_match = re.search(r'(\d+)(?:st|nd|rd|th)-level', line)
            if level_match:
                level = int(level_match.group(1))
            else:
                level = -1  # Unknown level

        # Extract school
        schools = ['abjuration', 'conjuration', 'divination', 'enchantment',
                   'evocation', 'illusion', 'necromancy', 'transmutation']

        school = 'unknown'
        for s in schools:
            if s in line:
                school = s
                break

        return level, school, ritual

    def _extract_field(self, block: str, field_name: str) -> str:
        """Extract a field value from the spell block"""
        block = self._normalize_ocr(block)
        pattern = rf'{field_name}:\s*([^\n]+)'
        match = re.search(pattern, block, re.IGNORECASE)
        return match.group(1).strip() if match else "Unknown"

    def _extract_description(self, block: str) -> str:
        """Extract the spell description (everything after Duration)"""
        block = self._normalize_ocr(block)
        duration_match = re.search(r'Duration:.*?\n', block, re.IGNORECASE | re.DOTALL)
        if not duration_match:
            return "Description not found"

        description_start = duration_match.end()
        description = block[description_start:].strip()
        return description


print("✅ SpellTextParser ready!")
print("   📄 Parses spells.txt with UPPERCASE names")
print("   🔍 Extracts complete metadata and descriptions")


✅ SpellTextParser ready!
   📄 Parses spells.txt with UPPERCASE names
   🔍 Extracts complete metadata and descriptions


In [20]:
# Cell 5: Spell Chunk Creator for RAG

class SpellChunkCreator:
    """Create optimized chunks for RAG retrieval"""

    def __init__(self, max_tokens: int = 400):
        self.max_tokens = max_tokens
        self.encoding = tiktoken.get_encoding("cl100k_base")

    def create_spell_chunks(self, metadata: SpellMetadata, description: str) -> List[SpellChunk]:
        """Create multiple chunks for a single spell"""
        chunks = []

        # 1. Core Mechanics Chunk (essential casting info)
        mechanics_chunk = self._create_mechanics_chunk(metadata, description)
        chunks.append(mechanics_chunk)

        # 2. Full Description Chunk (if description is substantial)
        if len(description) > 100:
            desc_chunk = self._create_description_chunk(metadata, description)
            chunks.append(desc_chunk)

        # 3. Tactical/Combat Chunk (if spell has combat applications)
        if self._is_combat_spell(metadata, description):
            combat_chunk = self._create_combat_chunk(metadata, description)
            if combat_chunk:
                chunks.append(combat_chunk)

        return chunks

    def _create_mechanics_chunk(self, metadata: SpellMetadata, description: str) -> SpellChunk:
        """Create core mechanics chunk - most important for RAG queries"""

        # Build concise mechanics summary
        content_parts = [
            f"**{metadata.name}**",
            f"Level {metadata.level} {metadata.school}",
        ]

        # Add class information if available
        if metadata.classes:
            content_parts.append(f"Classes: {', '.join(metadata.classes)}")

        # Core casting info
        content_parts.extend([
            f"Casting Time: {metadata.casting_time}",
            f"Range: {metadata.range}",
            f"Components: {metadata.components}",
            f"Duration: {metadata.duration}"
        ])

        # Special properties
        if metadata.ritual:
            content_parts.append("⭐ Can be cast as ritual")
        if metadata.concentration:
            content_parts.append("🎯 Requires concentration")

        # Add first sentence or two of description for context
        desc_preview = self._get_description_preview(description)
        if desc_preview:
            content_parts.append(f"\n{desc_preview}")

        content = "\n".join(content_parts)

        # Generate tags for retrieval
        tags = self._generate_spell_tags(metadata, description)

        return SpellChunk(
            spell_name=metadata.name,
            content=content,
            chunk_type="mechanics",
            metadata=metadata,
            tags=tags
        )

    def _create_description_chunk(self, metadata: SpellMetadata, description: str) -> SpellChunk:
        """Create full description chunk"""

        # Truncate if too long
        max_desc_length = self.max_tokens * 3  # Rough character limit
        if len(description) > max_desc_length:
            description = description[:max_desc_length] + "..."

        content = f"**{metadata.name} - Full Description**\n{description}"

        tags = self._generate_spell_tags(metadata, description)
        tags.add("full_description")

        return SpellChunk(
            spell_name=metadata.name,
            content=content,
            chunk_type="description",
            metadata=metadata,
            tags=tags
        )

    def _create_combat_chunk(self, metadata: SpellMetadata, description: str) -> Optional[SpellChunk]:
        """Create combat-focused chunk if spell has combat applications"""

        combat_keywords = self._extract_combat_info(description)
        if not combat_keywords:
            return None

        content_parts = [
            f"**{metadata.name} - Combat Applications**",
            f"Level {metadata.level} | {metadata.casting_time} | {metadata.range}",
        ]

        # Add relevant combat excerpts
        combat_text = self._extract_combat_relevant_text(description)
        if combat_text:
            content_parts.append(combat_text)

        content = "\n".join(content_parts)

        tags = self._generate_spell_tags(metadata, description)
        tags.update(["combat", "tactical"])

        return SpellChunk(
            spell_name=metadata.name,
            content=content,
            chunk_type="combat",
            metadata=metadata,
            tags=tags
        )

    def _is_combat_spell(self, metadata: SpellMetadata, description: str) -> bool:
        """Check if spell has combat applications"""
        combat_indicators = [
            'damage', 'attack', 'hit points', 'armor class', 'saving throw',
            'weapon', 'target', 'creatures', 'enemy', 'hostile'
        ]

        desc_lower = description.lower()
        return any(indicator in desc_lower for indicator in combat_indicators)

    def _get_description_preview(self, description: str, max_sentences: int = 2) -> str:
        """Get first few sentences of description"""
        sentences = re.split(r'[.!?]+', description)
        preview_sentences = []

        for sentence in sentences[:max_sentences]:
            sentence = sentence.strip()
            if sentence and len(sentence) > 10:
                preview_sentences.append(sentence)

        return '. '.join(preview_sentences) + '.' if preview_sentences else ""

    def _extract_combat_info(self, description: str) -> List[str]:
        """Extract combat-relevant keywords"""
        combat_patterns = [
            r'(\d+d\d+(?:\s*[+\-]\s*\d+)?)',  # Dice notation
            r'(DC \d+)',  # Difficulty Class
            r'(armor class|AC)',  # AC references
            r'(saving throw)',  # Saves
            r'(\d+\s*feet?)',  # Distances
        ]

        keywords = []
        for pattern in combat_patterns:
            matches = re.findall(pattern, description, re.IGNORECASE)
            keywords.extend(matches)

        return keywords

    def _extract_combat_relevant_text(self, description: str) -> str:
        """Extract sentences most relevant to combat"""
        sentences = re.split(r'[.!?]+', description)
        combat_sentences = []

        combat_keywords = [
            'damage', 'attack', 'hit', 'save', 'saving throw', 'dc',
            'armor class', 'target', 'creature', 'spell attack'
        ]

        for sentence in sentences:
            sentence = sentence.strip()
            if any(keyword in sentence.lower() for keyword in combat_keywords):
                combat_sentences.append(sentence)

        return '. '.join(combat_sentences[:3]) + '.' if combat_sentences else ""

    def _generate_spell_tags(self, metadata: SpellMetadata, description: str) -> Set[str]:
        """Generate tags for retrieval optimization"""
        tags = set()

        # Basic tags
        tags.add(f"level_{metadata.level}")
        tags.add(f"school_{metadata.school}")

        # Class tags
        for spell_class in metadata.classes:
            tags.add(f"class_{spell_class.lower()}")

        # Special properties
        if metadata.ritual:
            tags.add("ritual")
        if metadata.concentration:
            tags.add("concentration")

        # Content-based tags
        desc_lower = description.lower()

        # Damage types
        damage_types = ['fire', 'cold', 'lightning', 'thunder', 'acid',
                       'poison', 'psychic', 'necrotic', 'radiant', 'force']
        for damage_type in damage_types:
            if damage_type in desc_lower:
                tags.add(f"damage_{damage_type}")
                tags.add("damage_spell")

        # Spell categories
        if any(word in desc_lower for word in ['heal', 'healing', 'restore', 'cure']):
            tags.add("healing")
        if any(word in desc_lower for word in ['charm', 'frighten', 'stun', 'paralyze']):
            tags.add("control")
        if any(word in desc_lower for word in ['summon', 'conjure', 'create']):
            tags.add("summoning")
        if any(word in desc_lower for word in ['teleport', 'dimension', 'plane']):
            tags.add("movement")

        return tags

print("✅ SpellChunkCreator ready!")
print("   📦 Creates optimized chunks for RAG retrieval")
print("   🎯 Mechanics, Description, and Combat chunks")
print("   🏷️ Auto-generates retrieval tags")

✅ SpellChunkCreator ready!
   📦 Creates optimized chunks for RAG retrieval
   🎯 Mechanics, Description, and Combat chunks
   🏷️ Auto-generates retrieval tags


In [21]:
# Cell 6: ChromaDB Manager

class SpellChromaManager:
    """ChromaDB vector database manager for spells"""

    def __init__(self, db_path: str = "./chromadb"):
        self.db_path = db_path
        self.client = chromadb.PersistentClient(path=db_path)

        try:
            self.collection = self.client.get_collection("spell_rag_v2")
            print(f"📂 Loaded existing database: {self.collection.count()} documents")
        except:
            self.collection = self.client.create_collection("spell_rag_v2")
            print("✨ Created new database")

    def add_spell_chunks(self, chunks: List[SpellChunk]):
        """Add spell chunks to database"""
        print(f"💾 Adding {len(chunks)} chunks...")

        documents = []
        metadatas = []
        ids = []

        for i, chunk in enumerate(chunks):
            ids.append(f"{chunk.spell_name}_{chunk.chunk_type}_{i}")
            documents.append(chunk.get_retrieval_text())

            metadata = chunk.metadata.to_dict()
            metadata["chunk_type"] = chunk.chunk_type
            metadatas.append(metadata)

        self.collection.add(
            documents=documents,
            metadatas=metadatas,
            ids=ids
        )
        print(f"✅ Added {len(chunks)} chunks")

    def search_spells(self, query: str, n_results: int = 5, filters: dict = None):
        """Search for spells"""
        results = self.collection.query(
            query_texts=[query],
            n_results=n_results
        )
        return results

    def get_spell_by_name(self, spell_name: str):
        """Get specific spell by name"""
        results = self.collection.query(
            query_texts=[f"{spell_name} spell"],
            n_results=20
        )

        # Filter results manually
        filtered = {"documents": [[]], "metadatas": [[]], "distances": [[]]}

        if results["documents"][0]:
            for i, meta in enumerate(results["metadatas"][0]):
                if spell_name.lower() in meta.get("spell_name", "").lower():
                    filtered["documents"][0].append(results["documents"][0][i])
                    filtered["metadatas"][0].append(meta)
                    filtered["distances"][0].append(results["distances"][0][i])

        return filtered

    def get_spells_by_class(self, class_name: str, level: int = None):
        """Get spells for a class"""
        query_text = f"{class_name} spells"
        if level is not None:
            query_text += f" level {level}"

        results = self.collection.query(
            query_texts=[query_text],
            n_results=50
        )
        return results

    def get_database_stats(self):
        """Get simple database stats"""
        try:
            count = self.collection.count()
            return {"total_documents": count}
        except:
            return {"total_documents": 0}

    def clear_database(self):
        """Clear database"""
        try:
            self.client.delete_collection("spell_rag_v2")
            self.collection = self.client.create_collection("spell_rag_v2")
            print("🗑️ Database cleared")
        except Exception as e:
            print(f"❌ Clear failed: {e}")

print("✅ SpellChromaManager ready!")

✅ SpellChromaManager ready!


In [22]:
# Cell 7: Main Spell Processing Pipeline

class SpellRAGPipeline:
    """Main pipeline orchestrating the entire spell processing workflow"""

    def __init__(self, db_path: str = "./chromadb"):
        self.list_processor = SpellListProcessor()
        self.text_parser = SpellTextParser()
        self.chunk_creator = SpellChunkCreator()
        self.chroma_manager = SpellChromaManager(db_path)

        print("🎲 SpellRAGPipeline initialized!")

    def process_all_spells(self, spells_txt_path: str, all_spells_copy_path: str,
                          clear_existing: bool = False) -> dict:
        """Complete pipeline: process both files and populate database"""

        print("🚀 Starting complete spell processing pipeline...")
        print("=" * 60)

        if clear_existing:
            print("🗑️ Clearing existing database...")
            self.chroma_manager.clear_database()

        # Step 1: Process class and level information
        print("\n📊 Step 1: Processing spell class and level data")
        try:
            with open(all_spells_copy_path, 'r', encoding='utf-8') as f:
                all_spells_text = f.read()
            self.list_processor.process_spell_lists(all_spells_text)
        except Exception as e:
            print(f"❌ Error processing spell lists: {e}")
            return {"success": False, "error": str(e)}

        # Step 2: Parse spell text and metadata
        print("\n📖 Step 2: Parsing spell descriptions and metadata")
        try:
            parsed_spells = self.text_parser.parse_spells_file(spells_txt_path)
        except Exception as e:
            print(f"❌ Error parsing spells: {e}")
            return {"success": False, "error": str(e)}

        # Step 3: Enhance with class/level info and create chunks
        print("\n🔗 Step 3: Enhancing spells with class info and creating chunks")
        all_chunks = []
        enhanced_count = 0

        for metadata, description in parsed_spells:
            # Enhance with class and level info
            level, classes = self.list_processor.get_spell_info(metadata.name)

            if level != -1:  # Found in class lists
                metadata.level = level
                enhanced_count += 1
            if classes:
                metadata.classes = classes

            # Create chunks for this spell
            chunks = self.chunk_creator.create_spell_chunks(metadata, description)
            all_chunks.extend(chunks)

        print(f"   🎯 Enhanced {enhanced_count}/{len(parsed_spells)} spells with class info")
        print(f"   📦 Created {len(all_chunks)} total chunks")

        # Step 4: Add to vector database
        print("\n💾 Step 4: Adding chunks to vector database")
        try:
            self.chroma_manager.add_spell_chunks(all_chunks)
        except Exception as e:
            print(f"❌ Error adding to database: {e}")
            return {"success": False, "error": str(e)}

        # Step 5: Generate summary
        print("\n📊 Step 5: Generating processing summary")
        stats = self.chroma_manager.get_database_stats()

        summary = {
            "success": True,
            "processed_spells": len(parsed_spells),
            "enhanced_spells": enhanced_count,
            "total_chunks": len(all_chunks),
            "database_stats": stats
        }

        print("\n" + "=" * 60)
        print("✅ PROCESSING COMPLETE!")
        print(f"📚 Processed {len(parsed_spells)} spells")
        print(f"🎯 Enhanced {enhanced_count} with class info")
        print(f"📦 Created {len(all_chunks)} chunks")
        print(f"🗄️ Database now contains {stats.get('total_documents', 0)} documents")

        return summary

    def search_spells(self, query: str, n_results: int = 5, filters: dict = None):
        """Search for spells and return formatted results"""
        print(f"🔍 Searching for: '{query}'")

        # Perform search
        results = self.chroma_manager.search_spells(query, n_results, filters)

        if not results["documents"][0]:
            print("❌ No results found")
            return []

        # Format results
        formatted_results = []
        for i, (doc, meta, distance) in enumerate(zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        )):
            formatted_result = {
                "rank": i + 1,
                "spell_name": meta.get("spell_name", "Unknown"),
                "chunk_type": meta.get("chunk_type", "unknown"),
                "level": meta.get("level", -1),
                "school": meta.get("school", "unknown"),
                "classes": meta.get("classes", "").split(",") if meta.get("classes") else [],
                "content": doc,
                "relevance_score": 1 - distance,  # Convert distance to similarity
                "metadata": meta
            }
            formatted_results.append(formatted_result)

        print(f"✅ Found {len(formatted_results)} results")
        return formatted_results

    def get_spell_details(self, spell_name: str) -> dict:
        """Get complete information for a specific spell"""
        print(f"📖 Getting details for: {spell_name}")

        results = self.chroma_manager.get_spell_by_name(spell_name)

        if not results["documents"]:
            print(f"❌ Spell '{spell_name}' not found")
            return {}

        # Organize chunks by type
        chunks_by_type = {}
        metadata = None

        for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
            chunk_type = meta.get("chunk_type", "unknown")
            chunks_by_type[chunk_type] = doc

            if not metadata:  # Store metadata from first chunk
                metadata = meta

        spell_details = {
            "spell_name": spell_name,
            "metadata": metadata,
            "chunks": chunks_by_type
        }

        print(f"✅ Found {len(chunks_by_type)} chunks for {spell_name}")
        return spell_details

    def get_class_spells(self, class_name: str, level: int = None) -> List[dict]:
        """Get spells for a specific class and level"""
        level_text = f" level {level}" if level is not None else ""
        print(f"📚 Getting {class_name}{level_text} spells")

        results = self.chroma_manager.get_spells_by_class(class_name, level)

        # Extract unique spells
        unique_spells = {}
        for meta in results["metadatas"][0]:
            spell_name = meta.get("spell_name")
            if spell_name and spell_name not in unique_spells:
                unique_spells[spell_name] = meta

        spell_list = list(unique_spells.values())
        spell_list.sort(key=lambda x: (x.get("level", 0), x.get("spell_name", "")))

        print(f"✅ Found {len(spell_list)} spells")
        return spell_list

print("✅ SpellRAGPipeline ready!")
print("   🎯 Complete orchestration of spell processing")
print("   🔍 Search and retrieval capabilities")
print("   📊 Comprehensive spell database management")

✅ SpellRAGPipeline ready!
   🎯 Complete orchestration of spell processing
   🔍 Search and retrieval capabilities
   📊 Comprehensive spell database management


In [23]:
# Cell 8: File Configuration and Validation

import os

def validate_files(spells_txt_path: str, all_spells_copy_path: str) -> dict:
    """Validate that required files exist and are readable"""

    print("📂 Validating input files...")

    validation_result = {
        "valid": True,
        "errors": [],
        "file_info": {}
    }

    # Check spells.txt
    if not os.path.exists(spells_txt_path):
        validation_result["valid"] = False
        validation_result["errors"].append(f"Spells file not found: {spells_txt_path}")
    else:
        try:
            with open(spells_txt_path, 'r', encoding='utf-8') as f:
                content = f.read()
            validation_result["file_info"]["spells_txt"] = {
                "path": spells_txt_path,
                "size": len(content),
                "sample": content[:200] + "..." if len(content) > 200 else content
            }
            print(f"✅ spells.txt: {len(content):,} characters")
        except Exception as e:
            validation_result["valid"] = False
            validation_result["errors"].append(f"Error reading spells.txt: {e}")

    # Check all_spells_copy.txt
    if not os.path.exists(all_spells_copy_path):
        validation_result["valid"] = False
        validation_result["errors"].append(f"All spells file not found: {all_spells_copy_path}")
    else:
        try:
            with open(all_spells_copy_path, 'r', encoding='utf-8') as f:
                content = f.read()
            validation_result["file_info"]["all_spells_copy"] = {
                "path": all_spells_copy_path,
                "size": len(content),
                "sample": content[:200] + "..." if len(content) > 200 else content
            }
            print(f"✅ all_spells_copy.txt: {len(content):,} characters")
        except Exception as e:
            validation_result["valid"] = False
            validation_result["errors"].append(f"Error reading all_spells_copy.txt: {e}")

    if validation_result["valid"]:
        print("✅ All files validated successfully!")
    else:
        print("❌ File validation failed:")
        for error in validation_result["errors"]:
            print(f"   • {error}")

    return validation_result

# File paths - UPDATE THESE TO MATCH YOUR ACTUAL FILE LOCATIONS
SPELLS_TXT_PATH = "spells.txt"  # Your main spells file with UPPERCASE names
ALL_SPELLS_COPY_PATH = "all_spells.txt"  # Your class/level mapping file

# Database configuration
CHROMA_DB_PATH = "./chromadb"

print("🎲 D&D 5e Spell RAG Configuration")
print("📁 File paths configured:")
print(f"   📖 Spells text: {SPELLS_TXT_PATH}")
print(f"   📊 Class data: {ALL_SPELLS_COPY_PATH}")
print(f"   🗄️ Database: {CHROMA_DB_PATH}")
print("\nTo validate files, run: validate_files(SPELLS_TXT_PATH, ALL_SPELLS_COPY_PATH)")

🎲 D&D 5e Spell RAG Configuration
📁 File paths configured:
   📖 Spells text: spells.txt
   📊 Class data: all_spells.txt
   🗄️ Database: ./chromadb

To validate files, run: validate_files(SPELLS_TXT_PATH, ALL_SPELLS_COPY_PATH)


In [24]:
# Cell 9: Main Processing and Testing

# Validate files first
validation = validate_files(SPELLS_TXT_PATH, ALL_SPELLS_COPY_PATH)

if not validation["valid"]:
    print("❌ Cannot proceed - please fix file path issues above")
    print("Update SPELLS_TXT_PATH and ALL_SPELLS_COPY_PATH in the previous cell")
else:
    print("✅ Files validated - ready to proceed!")

    # Initialize the pipeline
    print("\n🚀 Initializing Spell RAG Pipeline...")
    pipeline = SpellRAGPipeline(db_path=CHROMA_DB_PATH)

    # Check if database already exists
    stats = pipeline.chroma_manager.get_database_stats()
    existing_docs = stats.get("total_documents", 0)

    if existing_docs > 0:
        print(f"\n📊 Existing database found with {existing_docs} documents")
        print("Options:")
        print("   1. Use existing database (skip processing)")
        print("   2. Clear and rebuild database")
        print("\nTo rebuild: run process_spells(clear_existing=True)")
        print("To use existing: proceed to testing below")
    else:
        print(f"\n💾 Processing spells and building database...")

        # Process all spells
        result = pipeline.process_all_spells(
            spells_txt_path=SPELLS_TXT_PATH,
            all_spells_copy_path=ALL_SPELLS_COPY_PATH,
            clear_existing=False
        )

        if result["success"]:
            print(f"\n🎉 SUCCESS! Database built with:")
            print(f"   📚 {result['processed_spells']} spells processed")
            print(f"   🎯 {result['enhanced_spells']} enhanced with class info")
            print(f"   📦 {result['total_chunks']} chunks created")
            print(f"   🗄️ {result['database_stats'].get('total_documents', 0)} documents stored")
        else:
            print(f"\n❌ Processing failed: {result.get('error', 'Unknown error')}")

📂 Validating input files...
✅ spells.txt: 355,225 characters
✅ all_spells_copy.txt: 13,455 characters
✅ All files validated successfully!
✅ Files validated - ready to proceed!

🚀 Initializing Spell RAG Pipeline...
✨ Created new database
🎲 SpellRAGPipeline initialized!

💾 Processing spells and building database...
🚀 Starting complete spell processing pipeline...

📊 Step 1: Processing spell class and level data
🔍 Processing spell class and level data...
   📖 Processing Bard spells...
   📖 Processing Cleric spells...
   📖 Processing Druid spells...
   📖 Processing Paladin spells...
   📖 Processing Ranger spells...
   📖 Processing Sorcerer spells...
   📖 Processing Warlock spells...
   📖 Processing Wizard spells...
✅ Processed 771 spell-class mappings
📊 Found 351 unique spells
📊 Found 351 spells with class info

📖 Step 2: Parsing spell descriptions and metadata
📖 Reading spells from: spells.txt
📄 Loaded 355,225 characters
🔍 Found 347 spell blocks
   ⚙️ Parsed 50/347 spells...
   ⚙️ Parsed 

In [25]:
# Cell 10: Testing and Example Usage

def run_spell_tests():
    """Run comprehensive tests of the spell RAG system"""

    print("🧪 Running Spell RAG Tests")
    print("=" * 50)

    # Test 1: Basic spell search
    print("\n🔍 Test 1: Basic spell search")
    results = pipeline.search_spells("fireball damage", n_results=3)

    for result in results:
        print(f"   📜 {result['spell_name']} (Level {result['level']} {result['school']})")
        print(f"      🎯 Relevance: {result['relevance_score']:.3f}")
        print(f"      📝 Type: {result['chunk_type']}")
        print(f"      🎭 Classes: {', '.join(result['classes'])}")
        print()

    # Test 2: Get specific spell details
    print("\n📖 Test 2: Get specific spell details")
    spell_details = pipeline.get_spell_details("Fireball")

    if spell_details:
        meta = spell_details["metadata"]
        print(f"   📜 {spell_details['spell_name']}")
        print(f"   🎯 Level {meta.get('level')} {meta.get('school')}")
        print(f"   ⏱️ {meta.get('casting_time')}")
        print(f"   📏 {meta.get('range')}")
        print(f"   🔮 {meta.get('components')}")
        print(f"   ⏳ {meta.get('duration')}")
        print(f"   🎭 Classes: {meta.get('classes', 'Unknown')}")
        print(f"   📦 Available chunks: {', '.join(spell_details['chunks'].keys())}")

    # Test 3: Class-specific spells
    print("\n📚 Test 3: Get Wizard spells (Level 3)")
    wizard_spells = pipeline.get_class_spells("Wizard", level=3)

    print(f"   Found {len(wizard_spells)} Level 3 Wizard spells:")
    for spell in wizard_spells[:5]:  # Show first 5
        print(f"   📜 {spell.get('spell_name', 'Unknown')}")
    if len(wizard_spells) > 5:
        print(f"   ... and {len(wizard_spells) - 5} more")

    # Test 4: Combat-focused search
    print("\n⚔️ Test 4: Combat spell search")
    combat_results = pipeline.search_spells("spell attack damage creatures", n_results=3)

    for result in combat_results:
        print(f"   ⚔️ {result['spell_name']} (Level {result['level']})")
        print(f"      🎯 Relevance: {result['relevance_score']:.3f}")
        print(f"      📝 {result['chunk_type']} chunk")
        print()

    # Test 5: Database statistics
    print("\n📊 Test 5: Database statistics")
    stats = pipeline.chroma_manager.get_database_stats()

    print(f"   📚 Total documents: {stats.get('total_documents', 0)}")
    print(f"   🧙 Unique spells: {stats.get('unique_spells', 0)}")
    print(f"   📦 Chunk types: {stats.get('chunk_types', {})}")
    print(f"   🎭 Classes: {len(stats.get('classes', []))}")
    print(f"   🏫 Schools: {stats.get('schools', {})}")

def search_examples():
    """Show example searches for different use cases"""

    print("🎯 Example Search Queries")
    print("=" * 40)

    example_queries = [
        ("Healing spells", "heal restore hit points"),
        ("Fire damage", "fire damage burning"),
        ("Control spells", "charm frighten stun paralyze"),
        ("Movement spells", "teleport fly speed movement"),
        ("Ritual spells", "ritual casting"),
        ("Concentration spells", "concentration duration"),
        ("Area of effect", "area radius sphere cone"),
        ("Saving throws", "saving throw dexterity constitution")
    ]

    for category, query in example_queries:
        print(f"\n🔍 {category}: '{query}'")
        results = pipeline.search_spells(query, n_results=2)

        for result in results[:2]:  # Show top 2
            print(f"   📜 {result['spell_name']} (Level {result['level']})")
            print(f"      🎯 Score: {result['relevance_score']:.3f}")

# Function to manually rebuild database if needed
def rebuild_database():
    """Rebuild the database from scratch"""
    print("🔄 Rebuilding database...")
    result = pipeline.process_all_spells(
        spells_txt_path=SPELLS_TXT_PATH,
        all_spells_copy_path=ALL_SPELLS_COPY_PATH,
        clear_existing=True
    )
    return result

print("🧪 Testing functions ready!")
print("\nAvailable functions:")
print("   🧪 run_spell_tests() - Comprehensive system tests")
print("   🎯 search_examples() - Show example search queries")
print("   🔄 rebuild_database() - Rebuild database from scratch")
print("\nExample usage:")
print("   results = pipeline.search_spells('fireball damage')")
print("   details = pipeline.get_spell_details('Fireball')")
print("   wizard_spells = pipeline.get_class_spells('Wizard', level=3)")

🧪 Testing functions ready!

Available functions:
   🧪 run_spell_tests() - Comprehensive system tests
   🎯 search_examples() - Show example search queries
   🔄 rebuild_database() - Rebuild database from scratch

Example usage:
   results = pipeline.search_spells('fireball damage')
   details = pipeline.get_spell_details('Fireball')
   wizard_spells = pipeline.get_class_spells('Wizard', level=3)


In [26]:
# Cell 11: Quick Start Demo

print("🎲 D&D 5e Spell RAG - Quick Demo")
print("=" * 40)

# Quick demonstration if database is ready
if 'pipeline' in globals():
    stats = pipeline.chroma_manager.get_database_stats()

    if stats.get("total_documents", 0) > 0:
        print(f"✅ Database ready with {stats['total_documents']} documents!")
        #print(f"📚 Covering {stats['unique_spells']} unique spells\n")

        # Quick demo searches
        demo_queries = [
            "healing spell for combat",
            "wizard fireball area damage",
            "teleportation magic"
        ]

        for query in demo_queries:
            print(f"🔍 Query: '{query}'")
            results = pipeline.search_spells(query, n_results=2)

            for i, result in enumerate(results, 1):
                print(f"   {i}. {result['spell_name']} (Lv{result['level']} {result['school']})")
                print(f"      Classes: {', '.join(result['classes'])}")
                print(f"      Score: {result['relevance_score']:.3f}")
            print()

        # Show a complete spell
        print("📖 Complete spell example - 'Aid':")
        aid_details = pipeline.get_spell_details("Aid")
        if aid_details and "mechanics" in aid_details["chunks"]:
            print(aid_details["chunks"]["mechanics"])
        else:
            print("Aid not found - trying Fireball...")
            fireball_details = pipeline.get_spell_details("Fireball")
            if fireball_details and "mechanics" in fireball_details["chunks"]:
                print(fireball_details["chunks"]["mechanics"])

    else:
        print("⚠️ Database is empty - run the processing cell first!")
        print("Update file paths in Cell 8, then run Cell 9")
else:
    print("⚠️ Pipeline not initialized - run previous cells first!")

print("\n🎯 System is ready for D&D text adventure integration!")
print("Next steps:")
print("   1. Test with: run_spell_tests()")
print("   2. Try searches: pipeline.search_spells('your query')")
print("   3. Get spell details: pipeline.get_spell_details('Spell Name')")
print("   4. Ready to integrate with your fine-tuned model!")

🎲 D&D 5e Spell RAG - Quick Demo
✅ Database ready with 957 documents!
🔍 Query: 'healing spell for combat'
🔍 Searching for: 'healing spell for combat'
✅ Found 2 results
   1. Cure Wounds (Lv1 evocation)
      Classes: Bard, Cleric, Druid, Paladin, Ranger
      Score: 0.285
   2. Healing Word (Lv1 evocation)
      Classes: Bard, Cleric, Druid
      Score: 0.262

🔍 Query: 'wizard fireball area damage'
🔍 Searching for: 'wizard fireball area damage'
✅ Found 2 results
   1. Fireball (Lv3 unknown)
      Classes: Sorcerer, Wizard
      Score: 0.120
   2. Fireball (Lv3 unknown)
      Classes: Sorcerer, Wizard
      Score: 0.119

🔍 Query: 'teleportation magic'
🔍 Searching for: 'teleportation magic'
✅ Found 2 results
   1. Teleport (Lv7 conjuration)
      Classes: Bard, Sorcerer, Wizard
      Score: 0.134
   2. Teleport (Lv7 conjuration)
      Classes: Bard, Sorcerer, Wizard
      Score: 0.071

📖 Complete spell example - 'Aid':
📖 Getting details for: Aid
✅ Found 3 chunks for Aid
**Aid** (mechanics

In [27]:
# Debug Cell: Investigate why only 75 spells found and 0 enhanced

print("🔍 DEBUGGING SPELL PARSING ISSUES")
print("=" * 50)

# First, let's examine the raw spells.txt format
print("\n📄 Step 1: Examining spells.txt format")
with open(SPELLS_TXT_PATH, 'r', encoding='utf-8') as f:
    content = f.read()

# Show first 2000 characters to see the structure
print(f"📋 First 2000 characters of spells.txt:")
print("-" * 40)
print(content[:2000])
print("-" * 40)

# Let's see what spell blocks were actually found
print(f"\n🔍 Step 2: Examining found spell blocks")
text_parser = SpellTextParser()
spell_blocks = text_parser._split_into_spell_blocks(content)

print(f"📊 Found {len(spell_blocks)} spell blocks")
print(f"📋 First 3 spell block headers:")

for i, block in enumerate(spell_blocks[:3]):
    lines = block.split('\n')
    header = lines[0] if lines else "No header"
    second_line = lines[1] if len(lines) > 1 else "No second line"
    print(f"   {i+1}. Header: '{header}'")
    print(f"      Second: '{second_line}'")
    print(f"      Length: {len(block)} chars")
    print()

# Now let's check the class mapping
print(f"\n📚 Step 3: Examining class mapping")
list_processor = SpellListProcessor()

with open(ALL_SPELLS_COPY_PATH, 'r', encoding='utf-8') as f:
    all_spells_content = f.read()

print(f"📋 First 1000 characters of all_spells_copy.txt:")
print("-" * 40)
print(all_spells_content[:1000])
print("-" * 40)

list_processor.process_spell_lists(all_spells_content)

print(f"\n📊 Class mapping results:")
print(f"   Found {len(list_processor.spell_level_mapping)} spells with levels")
print(f"   Found {len(list_processor.class_spell_mapping)} spells with classes")

print(f"\n📋 First 10 spells from class mapping:")
for i, (spell_name, level) in enumerate(list(list_processor.spell_level_mapping.items())[:10]):
    classes = list_processor.class_spell_mapping.get(spell_name, [])
    print(f"   {i+1}. '{spell_name}' - Level {level} - Classes: {classes}")

print(f"\n🔍 Step 4: Testing name matching")
# Parse a few spells and see if names match
parsed_spells = []
for block in spell_blocks[:10]:  # Test first 10
    try:
        metadata, description = text_parser._parse_spell_block(block)
        parsed_spells.append(metadata.name)
        level, classes = list_processor.get_spell_info(metadata.name)
        print(f"   Parsed: '{metadata.name}' -> Level: {level}, Classes: {classes}")
    except Exception as e:
        print(f"   Parse failed: {e}")

print(f"\n📊 SUMMARY:")
print(f"   Spells.txt blocks found: {len(spell_blocks)}")
print(f"   Class file spells: {len(list_processor.spell_level_mapping)}")
print(f"   Successfully parsed: {len(parsed_spells)}")
print(f"   This suggests the issue is in spell block detection regex")

🔍 DEBUGGING SPELL PARSING ISSUES

📄 Step 1: Examining spells.txt format
📋 First 2000 characters of spells.txt:
----------------------------------------
ACID SPLASH
Conjuration calltrip
Casting Time: I action
Range: 60 feel
Components: V, S
Duration: Instantaneous
Vou hurl a bubble of acid. Choose one creature within
range, or choose two crealures within range thal are
wilhin 5 feet of each other. A target must succeed on a
Dexterity saving throw or lake Id6 acid damage.
This spell's damage increases by Id6 when you reach
5th leveI (2d6), 11th leveI (3d6), and 17lh leveI (4d6).
AID
2nd-leveI abjuration
Casting Time: I action
Range: 30 feet
Components: V, S, M (a tiny strip of while clOlh)
Duration: 8 hours
Your spell bolsters your allies with toughness and
resolve. Choose up to three crealures within range.
Each target's hit poinl maximum and current hit points
increase by 5 for the duration.
At Higher LeveIs. When you cast this spell using
a spell slot of 3rd leveI or higher, a largel'

In [34]:
import chromadb

# Reconnect to your persisted Chroma
client = chromadb.PersistentClient(path="./chromadb")

# Load your spells collection
collection = client.get_or_create_collection("spell_rag_v2")

# Now query for Aid
results = collection.query(
    query_texts=["Aid"],
    n_results=5
)

print("✅ Query Results for 'Aid':")
for i, doc in enumerate(results["documents"][0]):
    meta = results["metadatas"][0][i]
    print(f"\nResult {i+1}:")
    print("Name:", meta.get("name"))
    print("Level:", meta.get("level"))
    print("School:", meta.get("school"))
    print("Text snippet:", doc[:200], "...")


✅ Query Results for 'Aid':

Result 1:
Name: None
Level: 2
School: abjuration
Text snippet: **Aid** (description)
**Aid - Full Description**
Your spell bolsters your allies with toughness and
resolve. Choose up to three creatures within range.
Each target's hit point maximum and current hit  ...

Result 2:
Name: None
Level: 2
School: abjuration
Text snippet: **Aid** (mechanics)
**Aid**
Level 2 abjuration
Classes: Cleric, Paladin
Casting Time: I action
Range: 30 feet
Components: V, S, M (a tiny strip of while clOlh)
Duration: 8 hours

Your spell bolsters y ...

Result 3:
Name: None
Level: 2
School: abjuration
Text snippet: **Aid** (combat)
**Aid - Combat Applications**
Level 2 | I action | 30 feet
Choose up to three creatures within range. Each target's hit point maximum and current hit points
increase by 5 for the dura ...

Result 4:
Name: None
Level: 1
School: evocation
Text snippet: **Cure Wounds** (combat)
**Cure Wounds - Combat Applications**
Level 1 | I action | Touch
A creature yo

In [35]:
import chromadb

client = chromadb.PersistentClient(path="./chromadb")
collection = client.get_or_create_collection("spell_rag_v2")

# Try a very broad query to see what's actually in there
results = collection.get(
    include=["documents", "metadatas"]  # Include both documents and metadata
)

if results['ids']:
    print("✅ All documents found:")
    for i, doc in enumerate(results["documents"]):
        meta = results["metadatas"][i]
        print(f"\n--- Document {i+1} ---")
        print(f"ID: {results['ids'][i]}")
        print(f"Name: {meta.get('name', 'N/A')}")
        print(f"Level: {meta.get('level', 'N/A')}")
        print(f"School: {meta.get('school', 'N/A')}")
        print(f"Text: {doc[:200]}..." if len(doc) > 200 else f"Text: {doc}")
else:
    print("❌ No documents found in the collection!")

✅ All documents found:

--- Document 1 ---
ID: Acid Splash_mechanics_0
Name: N/A
Level: 0
School: conjuration
Text: **Acid Splash** (mechanics)
**Acid Splash**
Level 0 conjuration
Classes: Sorcerer, Wizard
Casting Time: I action
Range: 60 feet
Components: V, S
Duration: Instantaneous

Vou hurl a bubble of acid. Cho...

--- Document 2 ---
ID: Acid Splash_description_1
Name: N/A
Level: 0
School: conjuration
Text: **Acid Splash** (description)
**Acid Splash - Full Description**
Vou hurl a bubble of acid. Choose one creature within
range, or choose two creatures within range thal are
within 5 feet of each other....

--- Document 3 ---
ID: Acid Splash_combat_2
Name: N/A
Level: 0
School: conjuration
Text: **Acid Splash** (combat)
**Acid Splash - Combat Applications**
Level 0 | I action | 60 feet
Choose one creature within
range, or choose two creatures within range thal are
within 5 feet of each other....

--- Document 4 ---
ID: Aid_mechanics_3
Name: N/A
Level: 2
School: abjuration
Text: **A

In [36]:
import chromadb

client = chromadb.PersistentClient(path="./chromadb")
collection = client.get_or_create_collection("spell_rag_v2")

# Check all collections
collections = client.list_collections()
print("All collections:", [col.name for col in collections])

# Check document count
count = collection.count()
print(f"Documents in 'spells': {count}")

# Try to get any data
data = collection.get()
print("Data retrieved:", data)

All collections: ['spell_rag_v2', 'spells']
Documents in 'spells': 957
Data retrieved: {'ids': ['Acid Splash_mechanics_0', 'Acid Splash_description_1', 'Acid Splash_combat_2', 'Aid_mechanics_3', 'Aid_description_4', 'Aid_combat_5', 'Alarm_mechanics_6', 'Alarm_description_7', 'Alarm_combat_8', 'Alter Self_mechanics_9', 'Alter Self_description_10', 'Alter Self_combat_11', 'Animal Friendship_mechanics_12', 'Animal Friendship_description_13', 'Animal Messenger_mechanics_14', 'Animal Messenger_description_15', 'Animate Dead_mechanics_16', 'Animate Dead_description_17', 'Animate Dead_combat_18', 'Animate Objects_mechanics_19', 'Animate Objects_description_20', 'Animate Objects_combat_21', 'Antilife Shell_mechanics_22', 'Antilife Shell_description_23', 'Antilife Shell_combat_24', 'Antimagic Field_mechanics_25', 'Antimagic Field_description_26', 'Antimagic Field_combat_27', 'Arcane Eye_mechanics_28', 'Arcane Eye_description_29', 'Arcane Gate_mechanics_30', 'Arcane Gate_description_31', 'Arcane

In [44]:
import re
import json
from typing import Dict, List, Optional, Set, Any
from dataclasses import dataclass, field, asdict
import chromadb

# New Monster Data Classes
@dataclass
class MonsterAbilityScores:
    strength: int
    dexterity: int
    constitution: int
    intelligence: int
    wisdom: int
    charisma: int

@dataclass
class MonsterAttack:
    name: str
    attack_bonus: int
    damage_dice: str
    damage_type: str
    range: Optional[str] = None
    description: Optional[str] = None

@dataclass
class MonsterCombatStats:
    armor_class: int
    hit_points: int
    hit_dice: str
    speed: Dict[str, str]
    ability_scores: MonsterAbilityScores
    saving_throws: Optional[Dict[str, int]] = None
    skills: Optional[Dict[str, int]] = None

@dataclass
class Monster:
    name: str
    size: str
    creature_type: str
    alignment: str
    challenge_rating: str
    combat_stats: MonsterCombatStats
    attacks: List[MonsterAttack]
    special_abilities: List[str]
    description: str
    environment: List[str]
    senses: Optional[str] = None
    languages: Optional[str] = None
    damage_resistances: Optional[List[str]] = None
    damage_immunities: Optional[List[str]] = None
    condition_immunities: Optional[List[str]] = None
    traits: Optional[List[str]] = None
    actions: Optional[List[str]] = None
    reactions: Optional[List[str]] = None
    legendary_actions: Optional[List[str]] = None

# Monster Chunk Classes for RAG
@dataclass
class MonsterChunk:
    """A chunk of monster content for RAG retrieval"""
    monster_name: str
    content: str
    chunk_type: str  # 'stats', 'combat', 'abilities', 'description', 'tactics'
    metadata: Dict[str, Any]
    tags: Set[str] = field(default_factory=set)

    def get_retrieval_text(self) -> str:
        """Get formatted text for embedding"""
        return f"**{self.monster_name}** ({self.chunk_type})\\n{self.content}"

class MonsterChunkCreator:
    """Create optimized chunks for monster RAG retrieval"""

    def __init__(self, max_tokens: int = 400):
        self.max_tokens = max_tokens

    def create_monster_chunks(self, monster: Monster) -> List[MonsterChunk]:
        """Create multiple chunks for a single monster"""
        chunks = []

        # Convert Monster to dict for metadata
        monster_dict = self._monster_to_dict(monster)

        # 1. Core Stats Chunk
        stats_chunk = self._create_stats_chunk(monster, monster_dict)
        chunks.append(stats_chunk)

        # 2. Combat Abilities Chunk
        combat_chunk = self._create_combat_chunk(monster, monster_dict)
        chunks.append(combat_chunk)

        # 3. Special Abilities Chunk
        if monster.special_abilities:
            abilities_chunk = self._create_abilities_chunk(monster, monster_dict)
            chunks.append(abilities_chunk)

        # 4. Description & Environment Chunk
        desc_chunk = self._create_description_chunk(monster, monster_dict)
        chunks.append(desc_chunk)

        # 5. Additional features if they exist
        if monster.traits:
            traits_chunk = self._create_traits_chunk(monster, monster_dict)
            chunks.append(traits_chunk)

        if monster.legendary_actions:
            legendary_chunk = self._create_legendary_chunk(monster, monster_dict)
            chunks.append(legendary_chunk)

        return chunks

    def _monster_to_dict(self, monster: Monster) -> Dict[str, Any]:
        """Convert Monster to dictionary for metadata"""
        return {
            "name": monster.name,
            "size": monster.size,
            "creature_type": monster.creature_type,
            "alignment": monster.alignment,
            "challenge_rating": monster.challenge_rating,
            "environment": monster.environment,
            "senses": monster.senses,
            "languages": monster.languages,
            "damage_resistances": monster.damage_resistances,
            "damage_immunities": monster.damage_immunities,
            "condition_immunities": monster.condition_immunities
        }

    def _create_stats_chunk(self, monster: Monster, monster_dict: dict) -> MonsterChunk:
        """Create core statistics chunk"""
        content_parts = [
            f"**{monster.name}**",
            f"{monster.size} {monster.creature_type}, {monster.alignment}",
            f"**Challenge Rating:** {monster.challenge_rating}",
            f"**Armor Class:** {monster.combat_stats.armor_class}",
            f"**Hit Points:** {monster.combat_stats.hit_points} ({monster.combat_stats.hit_dice})",
            f"**Speed:** {', '.join([f'{k}: {v}' for k, v in monster.combat_stats.speed.items()])}",
        ]

        # Add ability scores
        abilities = monster.combat_stats.ability_scores
        content_parts.append(
            f"**Abilities:** STR {abilities.strength}, DEX {abilities.dexterity}, CON {abilities.constitution}, "
            f"INT {abilities.intelligence}, WIS {abilities.wisdom}, CHA {abilities.charisma}"
        )

        # Add saving throws if available
        if monster.combat_stats.saving_throws:
            saves = ", ".join([f"{k} {v}" for k, v in monster.combat_stats.saving_throws.items()])
            content_parts.append(f"**Saving Throws:** {saves}")

        # Add skills if available
        if monster.combat_stats.skills:
            skills = ", ".join([f"{k} {v}" for k, v in monster.combat_stats.skills.items()])
            content_parts.append(f"**Skills:** {skills}")

        # Add senses and languages
        if monster.senses:
            content_parts.append(f"**Senses:** {monster.senses}")
        if monster.languages:
            content_parts.append(f"**Languages:** {monster.languages}")

        content = "\\n".join(content_parts)

        # Generate tags
        tags = self._generate_monster_tags(monster)
        tags.add("stats")
        tags.add("core")

        return MonsterChunk(
            monster_name=monster.name,
            content=content,
            chunk_type="stats",
            metadata=monster_dict,
            tags=tags
        )

    def _create_combat_chunk(self, monster: Monster, monster_dict: dict) -> MonsterChunk:
        """Create combat-focused chunk"""
        content_parts = [
            f"**{monster.name} - Combat**",
            f"CR {monster.challenge_rating}, AC {monster.combat_stats.armor_class}, HP {monster.combat_stats.hit_points}"
        ]

        # Add attacks
        if monster.attacks:
            content_parts.append("**Attacks:**")
            for attack in monster.attacks:
                attack_desc = f"- **{attack.name}:** +{attack.attack_bonus} to hit, {attack.damage_dice} {attack.damage_type}"
                if attack.range:
                    attack_desc += f" ({attack.range})"
                if attack.description:
                    attack_desc += f". {attack.description}"
                content_parts.append(attack_desc)

        # Add damage resistances/immunities
        if monster.damage_resistances:
            content_parts.append(f"**Damage Resistances:** {', '.join(monster.damage_resistances)}")
        if monster.damage_immunities:
            content_parts.append(f"**Damage Immunities:** {', '.join(monster.damage_immunities)}")
        if monster.condition_immunities:
            content_parts.append(f"**Condition Immunities:** {', '.join(monster.condition_immunities)}")

        content = "\\n".join(content_parts)

        tags = self._generate_monster_tags(monster)
        tags.update(["combat", "attacks", "damage"])

        return MonsterChunk(
            monster_name=monster.name,
            content=content,
            chunk_type="combat",
            metadata=monster_dict,
            tags=tags
        )

    def _create_abilities_chunk(self, monster: Monster, monster_dict: dict) -> MonsterChunk:
        """Create special abilities chunk"""
        content_parts = [
            f"**{monster.name} - Special Abilities**",
            f"CR {monster.challenge_rating}"
        ]

        if monster.special_abilities:
            content_parts.append("**Special Abilities:**")
            for ability in monster.special_abilities:
                content_parts.append(f"- {ability}")

        content = "\\n".join(content_parts)

        tags = self._generate_monster_tags(monster)
        tags.add("abilities")
        tags.add("special")

        return MonsterChunk(
            monster_name=monster.name,
            content=content,
            chunk_type="abilities",
            metadata=monster_dict,
            tags=tags
        )

    def _create_description_chunk(self, monster: Monster, monster_dict: dict) -> MonsterChunk:
        """Create description and environment chunk"""
        content_parts = [
            f"**{monster.name} - Description & Environment**",
            f"{monster.size} {monster.creature_type}, {monster.alignment}"
        ]

        if monster.description:
            # Truncate if too long
            desc = monster.description
            if len(desc) > 300:
                desc = desc[:300] + "..."
            content_parts.append(f"**Description:** {desc}")

        if monster.environment:
            content_parts.append(f"**Environment:** {', '.join(monster.environment)}")

        content = "\\n".join(content_parts)

        tags = self._generate_monster_tags(monster)
        tags.update(["description", "environment", "lore"])

        return MonsterChunk(
            monster_name=monster.name,
            content=content,
            chunk_type="description",
            metadata=monster_dict,
            tags=tags
        )

    def _create_traits_chunk(self, monster: Monster, monster_dict: dict) -> MonsterChunk:
        """Create traits chunk"""
        content_parts = [
            f"**{monster.name} - Traits**",
            f"CR {monster.challenge_rating}"
        ]

        if monster.traits:
            content_parts.append("**Traits:**")
            for trait in monster.traits:
                content_parts.append(f"- {trait}")

        content = "\\n".join(content_parts)

        tags = self._generate_monster_tags(monster)
        tags.add("traits")

        return MonsterChunk(
            monster_name=monster.name,
            content=content,
            chunk_type="traits",
            metadata=monster_dict,
            tags=tags
        )

    def _create_legendary_chunk(self, monster: Monster, monster_dict: dict) -> MonsterChunk:
        """Create legendary actions chunk"""
        content_parts = [
            f"**{monster.name} - Legendary Actions**",
            f"CR {monster.challenge_rating}"
        ]

        if monster.legendary_actions:
            content_parts.append("**Legendary Actions:**")
            for action in monster.legendary_actions:
                content_parts.append(f"- {action}")

        content = "\\n".join(content_parts)

        tags = self._generate_monster_tags(monster)
        tags.add("legendary")
        tags.add("actions")

        return MonsterChunk(
            monster_name=monster.name,
            content=content,
            chunk_type="legendary",
            metadata=monster_dict,
            tags=tags
        )

    def _generate_monster_tags(self, monster: Monster) -> Set[str]:
        """Generate tags for retrieval optimization"""
        tags = set()

        # Basic tags
        tags.add(f"cr_{monster.challenge_rating}")
        tags.add(f"size_{monster.size.lower()}")
        tags.add(f"type_{monster.creature_type.lower().replace(' ', '_')}")

        # Alignment tags
        alignment = monster.alignment.lower()
        if 'lawful' in alignment:
            tags.add("alignment_lawful")
        if 'chaotic' in alignment:
            tags.add("alignment_chaotic")
        if 'good' in alignment:
            tags.add("alignment_good")
        if 'evil' in alignment:
            tags.add("alignment_evil")
        if 'neutral' in alignment:
            tags.add("alignment_neutral")

        # Environment tags
        if monster.environment:
            for env in monster.environment:
                env_clean = env.lower().replace(" ", "_").replace(",", "")
                tags.add(f"env_{env_clean}")

        # Combat role tags (infer from stats)
        abilities = monster.combat_stats.ability_scores
        if abilities.strength >= 18:
            tags.add("role_melee")
            tags.add("strength_based")
        if abilities.dexterity >= 16:
            tags.add("role_ranged")
            tags.add("dexterity_based")
        if abilities.intelligence >= 16 or abilities.charisma >= 16:
            tags.add("role_caster")
            tags.add("spellcaster")
        if abilities.constitution >= 16:
            tags.add("tank")

        # Damage type tags
        if monster.attacks:
            for attack in monster.attacks:
                dmg_type = attack.damage_type.lower().replace(" ", "_")
                tags.add(f"damage_{dmg_type}")

        return tags

# ChromaDB Manager for Monsters
class MonsterChromaManager:
    """ChromaDB manager for monster collections"""

    def __init__(self, db_path: str = "./chromadb"):
        self.db_path = db_path
        self.client = chromadb.PersistentClient(path=db_path)
        self.collection = self.client.get_or_create_collection("monsters")

    def add_monster_chunks(self, chunks: List[MonsterChunk]):
        """Add monster chunks to database with ChromaDB-compatible metadata"""
        documents = []
        metadatas = []
        ids = []

        for i, chunk in enumerate(chunks):
            chunk_id = f"{chunk.monster_name.lower().replace(' ', '_')}_{chunk.chunk_type}_{i}"
            ids.append(chunk_id)
            documents.append(chunk.get_retrieval_text())

            # Convert metadata to ChromaDB-compatible format
            metadata = {}
            for key, value in chunk.metadata.items():
                if isinstance(value, list):
                    metadata[key] = ", ".join(str(item) for item in value)
                elif value is None:
                    metadata[key] = "Unknown"
                else:
                    metadata[key] = value

            metadata["chunk_type"] = chunk.chunk_type
            metadata["monster_name"] = chunk.monster_name
            metadata["tags"] = ", ".join(chunk.tags)

            metadatas.append(metadata)

        self.collection.add(
            documents=documents,
            metadatas=metadatas,
            ids=ids
        )
    def search_monsters(self, query: str, n_results: int = 5, filters: dict = None):
        """Search monsters with optional filters"""
        if filters:
            results = self.collection.query(
                query_texts=[query],
                n_results=n_results,
                where=filters
            )
        else:
            results = self.collection.query(
                query_texts=[query],
                n_results=n_results
            )
        return results

    def get_collection_stats(self):
        """Get collection statistics"""
        return {
            "total_documents": self.collection.count(),
            "collection_name": "monsters"
        }

# Example usage with sample monster data
def create_sample_monster() -> Monster:
    """Create a sample monster for testing"""
    return Monster(
        name="Ancient Red Dragon",
        size="Gargantuan",
        creature_type="Dragon",
        alignment="Chaotic Evil",
        challenge_rating="24",
        combat_stats=MonsterCombatStats(
            armor_class=22,
            hit_points=546,
            hit_dice="28d20 + 252",
            speed={"walk": "40 ft", "fly": "80 ft", "burrow": "40 ft"},
            ability_scores=MonsterAbilityScores(
                strength=30, dexterity=10, constitution=29,
                intelligence=18, wisdom=15, charisma=23
            ),
            saving_throws={"Dex": 9, "Con": 16, "Wis": 10, "Cha": 14}
        ),
        attacks=[
            MonsterAttack(
                name="Bite",
                attack_bonus=17,
                damage_dice="2d10 + 10",
                damage_type="piercing",
                description="Plus 7d6 fire damage"
            ),
            MonsterAttack(
                name="Claw",
                attack_bonus=17,
                damage_dice="2d6 + 10",
                damage_type="slashing"
            )
        ],
        special_abilities=[
            "Fire Breath: 90-ft cone, 26d6 fire damage, DC 24 Dex save for half",
            "Legendary Resistance (3/day): Succeed on failed saving throw",
            "Frightful Presence: DC 21 Wis save or frightened for 1 minute"
        ],
        description="An ancient red dragon is the most powerful of the chromatic dragons...",
        environment=["Mountain", "Volcanic", "Underground"],
        senses="blindsight 60 ft., darkvision 120 ft.",
        languages="Common, Draconic",
        damage_immunities=["fire"],
        legendary_actions=[
            "Detect: Make a Wisdom (Perception) check",
            "Tail Attack: 2d6 + 10 slashing damage",
            "Wing Attack: 2d6 + 10 bludgeoning damage and knock prone"
        ]
    )

# Test the system


In [45]:
import re
import pdfplumber
from typing import Dict, List, Optional, Set, Any, Tuple
from dataclasses import dataclass, field, asdict
import chromadb

# Monster Data Classes (from previous code)
@dataclass
class MonsterAbilityScores:
    strength: int
    dexterity: int
    constitution: int
    intelligence: int
    wisdom: int
    charisma: int

@dataclass
class MonsterAttack:
    name: str
    attack_bonus: int
    damage_dice: str
    damage_type: str
    range: Optional[str] = None
    description: Optional[str] = None

@dataclass
class MonsterCombatStats:
    armor_class: int
    hit_points: int
    hit_dice: str
    speed: Dict[str, str]
    ability_scores: MonsterAbilityScores
    saving_throws: Optional[Dict[str, int]] = None
    skills: Optional[Dict[str, int]] = None

@dataclass
class Monster:
    name: str
    size: str
    creature_type: str
    alignment: str
    challenge_rating: str
    combat_stats: MonsterCombatStats
    attacks: List[MonsterAttack]
    special_abilities: List[str]
    description: str
    environment: List[str]
    senses: Optional[str] = None
    languages: Optional[str] = None
    damage_resistances: Optional[List[str]] = None
    damage_immunities: Optional[List[str]] = None
    condition_immunities: Optional[List[str]] = None
    traits: Optional[List[str]] = None
    actions: Optional[List[str]] = None
    reactions: Optional[List[str]] = None
    legendary_actions: Optional[List[str]] = None

# PDF Parser for Monster Manual
class MonsterManualPDFParser:
    """Parse D&D 5e Monster Manual PDF text into Monster objects"""

    def __init__(self):
        self.monster_patterns = {
            'name': r'^([A-Z][A-Za-z\s\'-]+)(?:\n|$)',
            'size_type': r'(Tiny|Small|Medium|Large|Huge|Gargantuan)\s+(aberration|beast|celestial|construct|dragon|elemental|fey|fiend|giant|humanoid|monstrosity|ooze|plant|undead)',
            'alignment': r',\s*([^,\n]+?)\s*(?:\n|$)',
            'ac': r'Armor Class\s*(\d+)',
            'hp': r'Hit Points\s*(\d+)\s*\(([^)]+)\)',
            'speed': r'Speed\s*([^\n]+)',
            'abilities': r'STR\s+(\d+)\s+DEX\s+(\d+)\s+CON\s+(\d+)\s+INT\s+(\d+)\s+WIS\s+(\d+)\s+CHA\s+(\d+)',
            'saves': r'Saving Throws\s*([^\n]+)',
            'skills': r'Skills\s*([^\n]+)',
            'senses': r'Senses\s*([^\n]+)',
            'languages': r'Languages\s*([^\n]+)',
            'cr': r'Challenge\s*(\d+(?:/\d+)?)',
            'traits': r'([A-Z][^.•]+?)\.\s*([^•]+)',
            'actions': r'Actions?[•:]\s*(.+?)(?=\n\n|\n[A-Z]|\Z)',
            'legendary': r'Legendary Actions?[•:]\s*(.+?)(?=\n\n|\n[A-Z]|\Z)'
        }

    def extract_pdf_text(self, pdf_path: str) -> Optional[str]:
        """Extract text from Monster Manual PDF"""
        try:
            print(f"📖 Extracting text from {pdf_path}...")
            full_text = ""

            with pdfplumber.open(pdf_path) as pdf:
                total_pages = len(pdf.pages)
                print(f"Processing {total_pages} pages...")

                for page_num, page in enumerate(pdf.pages, 1):
                    if page_num % 50 == 0:
                        print(f"Processed {page_num}/{total_pages} pages...")

                    page_text = page.extract_text()
                    if page_text:
                        full_text += page_text + "\n\n"

            print(f"✅ Extraction complete! {len(full_text):,} characters")
            return full_text

        except Exception as e:
            print(f"❌ PDF extraction failed: {e}")
            return None

    def split_into_monster_blocks(self, text: str) -> List[str]:
        """Split text into individual monster stat blocks"""
        # Look for monster names in all caps as section starters
        monster_blocks = []
        current_block = []
        lines = text.split('\n')

        for line in lines:
            line = line.strip()
            if not line:
                continue

            # Check if this line starts a new monster (all caps, reasonable length)
            if (line.isupper() and 2 <= len(line) <= 50 and
                not any(word in line for word in ['CONTENTS', 'CHAPTER', 'INDEX', 'CREDITS'])):
                if current_block:
                    monster_blocks.append('\n'.join(current_block))
                    current_block = []

            current_block.append(line)

        if current_block:
            monster_blocks.append('\n'.join(current_block))

        print(f"📦 Found {len(monster_blocks)} potential monster blocks")
        return monster_blocks

    def parse_monster_block(self, block_text: str) -> Optional[Monster]:
        """Parse a single monster stat block"""
        try:
            lines = block_text.split('\n')
            if len(lines) < 10:  # Too short to be a real monster
                return None

            # Extract basic info
            name = self._extract_monster_name(lines)
            if not name:
                return None

            size, creature_type = self._extract_size_type(block_text)
            alignment = self._extract_alignment(block_text)
            challenge_rating = self._extract_challenge_rating(block_text)

            # Extract combat stats
            armor_class = self._extract_armor_class(block_text)
            hit_points, hit_dice = self._extract_hit_points(block_text)
            speed = self._extract_speed(block_text)
            ability_scores = self._extract_ability_scores(block_text)

            # Extract additional info
            saving_throws = self._extract_saving_throws(block_text)
            skills = self._extract_skills(block_text)
            senses = self._extract_senses(block_text)
            languages = self._extract_languages(block_text)

            # Extract attacks and abilities
            attacks = self._extract_attacks(block_text)
            special_abilities = self._extract_special_abilities(block_text)
            traits = self._extract_traits(block_text)
            legendary_actions = self._extract_legendary_actions(block_text)

            # Extract description (first paragraph after stats)
            description = self._extract_description(block_text)
            environment = self._extract_environment(block_text)

            # Create combat stats
            combat_stats = MonsterCombatStats(
                armor_class=armor_class or 10,
                hit_points=hit_points or 1,
                hit_dice=hit_dice or "1d8",
                speed=speed or {"walk": "30 ft"},
                ability_scores=ability_scores or MonsterAbilityScores(10, 10, 10, 10, 10, 10),
                saving_throws=saving_throws,
                skills=skills
            )

            return Monster(
                name=name,
                size=size or "Medium",
                creature_type=creature_type or "humanoid",
                alignment=alignment or "unaligned",
                challenge_rating=challenge_rating or "1",
                combat_stats=combat_stats,
                attacks=attacks,
                special_abilities=special_abilities,
                description=description or "A mysterious creature.",
                environment=environment or ["Any"],
                senses=senses,
                languages=languages,
                traits=traits,
                legendary_actions=legendary_actions
            )

        except Exception as e:
            print(f"⚠️ Error parsing monster block: {e}")
            return None

    # Helper extraction methods (simplified versions)
    def _extract_monster_name(self, lines: List[str]) -> Optional[str]:
        """Extract monster name from first few lines"""
        for line in lines[:3]:
            line = line.strip()
            if (line.isupper() and 2 <= len(line) <= 50 and
                not any(word in line for word in ['CONTENTS', 'CHAPTER', 'PAGE'])):
                return line.title()
        return None

    def _extract_size_type(self, text: str) -> Tuple[Optional[str], Optional[str]]:
        """Extract size and creature type"""
        match = re.search(self.monster_patterns['size_type'], text, re.IGNORECASE)
        if match:
            return match.group(1), match.group(2)
        return None, None

    def _extract_armor_class(self, text: str) -> Optional[int]:
        """Extract armor class"""
        match = re.search(self.monster_patterns['ac'], text, re.IGNORECASE)
        return int(match.group(1)) if match else None

    def _extract_hit_points(self, text: str) -> Tuple[Optional[int], Optional[str]]:
        """Extract hit points and hit dice"""
        match = re.search(self.monster_patterns['hp'], text, re.IGNORECASE)
        if match:
            return int(match.group(1)), match.group(2)
        return None, None

    def _extract_ability_scores(self, text: str) -> Optional[MonsterAbilityScores]:
        """Extract ability scores"""
        match = re.search(self.monster_patterns['abilities'], text, re.IGNORECASE)
        if match:
            return MonsterAbilityScores(
                strength=int(match.group(1)),
                dexterity=int(match.group(2)),
                constitution=int(match.group(3)),
                intelligence=int(match.group(4)),
                wisdom=int(match.group(5)),
                charisma=int(match.group(6))
            )
        return None

    def _extract_alignment(self, text: str) -> Optional[str]:
        """Extract alignment from text"""
        # Look for alignment patterns
        alignment_patterns = [
            r',\s*((?:chaotic|lawful|neutral|good|evil|unaligned|\s)+)\s*(?:\n|$)',
            r'Alignment\s*[:\•]\s*([^\n]+)',
            r',\s*(chaotic good|chaotic evil|chaotic neutral|lawful good|lawful evil|lawful neutral|neutral good|neutral evil|true neutral|unaligned)'
        ]

        for pattern in alignment_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                alignment = match.group(1).strip()
                # Clean up common issues
                alignment = re.sub(r'[^a-zA-Z\s]', '', alignment)
                return alignment.title()

        return "Unaligned"

    def _extract_challenge_rating(self, text: str) -> Optional[str]:
        """Extract challenge rating"""
        cr_patterns = [
            r'Challenge\s*(\d+(?:/\d+)?)',
            r'CR\s*(\d+(?:/\d+)?)',
            r'Challenge Rating\s*[:\•]\s*(\d+(?:/\d+)?)'
        ]

        for pattern in cr_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                return match.group(1)

        return "1"  # Default to CR 1

    def _extract_speed(self, text: str) -> Dict[str, str]:
        """Extract speed information"""
        speed_match = re.search(r'Speed\s*([^\n]+)', text, re.IGNORECASE)
        if not speed_match:
            return {"walk": "30 ft"}

        speed_text = speed_match.group(1)
        speed_dict = {}

        # Parse different movement types
        patterns = {
            'walk': r'(\d+)\s*ft',
            'fly': r'fly\s*(\d+)\s*ft',
            'swim': r'swim\s*(\d+)\s*ft',
            'burrow': r'burrow\s*(\d+)\s*ft',
            'climb': r'climb\s*(\d+)\s*ft'
        }

        for move_type, pattern in patterns.items():
            match = re.search(pattern, speed_text, re.IGNORECASE)
            if match:
                speed_dict[move_type] = f"{match.group(1)} ft"

        return speed_dict if speed_dict else {"walk": "30 ft"}

    def _extract_saving_throws(self, text: str) -> Optional[Dict[str, int]]:
        """Extract saving throws"""
        saves_match = re.search(r'Saving Throws\s*([^\n]+)', text, re.IGNORECASE)
        if not saves_match:
            return None

        saves_text = saves_match.group(1)
        saves_dict = {}

        # Look for patterns like "Str +5, Dex +3"
        save_matches = re.findall(r'(\w+)\s*([+-]?\d+)', saves_text)
        for ability, bonus in save_matches:
            ability_lower = ability.lower()
            if ability_lower in ['str', 'strength']:
                saves_dict['Strength'] = int(bonus)
            elif ability_lower in ['dex', 'dexterity']:
                saves_dict['Dexterity'] = int(bonus)
            elif ability_lower in ['con', 'constitution']:
                saves_dict['Constitution'] = int(bonus)
            elif ability_lower in ['int', 'intelligence']:
                saves_dict['Intelligence'] = int(bonus)
            elif ability_lower in ['wis', 'wisdom']:
                saves_dict['Wisdom'] = int(bonus)
            elif ability_lower in ['cha', 'charisma']:
                saves_dict['Charisma'] = int(bonus)

        return saves_dict if saves_dict else None

    def _extract_skills(self, text: str) -> Optional[Dict[str, int]]:
        """Extract skills"""
        skills_match = re.search(r'Skills\s*([^\n]+)', text, re.IGNORECASE)
        if not skills_match:
            return None

        skills_text = skills_match.group(1)
        skills_dict = {}

        # Look for patterns like "Perception +5, Stealth +3"
        skill_matches = re.findall(r'([A-Za-z\s]+)\s*([+-]?\d+)', skills_text)
        for skill, bonus in skill_matches:
            skill_clean = skill.strip().title()
            skills_dict[skill_clean] = int(bonus)

        return skills_dict if skills_dict else None

    def _extract_senses(self, text: str) -> Optional[str]:
        """Extract senses"""
        senses_match = re.search(r'Senses\s*([^\n]+)', text, re.IGNORECASE)
        return senses_match.group(1).strip() if senses_match else None

    def _extract_languages(self, text: str) -> Optional[str]:
        """Extract languages"""
        languages_match = re.search(r'Languages\s*([^\n]+)', text, re.IGNORECASE)
        return languages_match.group(1).strip() if languages_match else None

    def _extract_traits(self, text: str) -> List[str]:
        """Extract traits"""
        traits = []
        # Look for trait sections
        trait_section = re.search(r'([A-Z][^.•]+?)\.\s*([^•]+)(?=\n\n|\n[A-Z]|\Z)', text)
        if trait_section:
            traits.append(f"{trait_section.group(1)}: {trait_section.group(2)}")
        return traits

    def _extract_legendary_actions(self, text: str) -> List[str]:
        """Extract legendary actions"""
        legendary_match = re.search(r'Legendary Actions?[:\•]\s*(.+?)(?=\n\n|\n[A-Z]|\Z)', text, re.IGNORECASE | re.DOTALL)
        if not legendary_match:
            return []

        legendary_text = legendary_match.group(1)
        actions = []

        # Split into individual actions
        action_matches = re.findall(r'•\s*([^•]+)', legendary_text)
        for action in action_matches:
            actions.append(action.strip())

        return actions
    def _extract_ability_scores(self, text: str) -> Optional[MonsterAbilityScores]:
        """Extract ability scores with better error handling"""
        # Method 1: Look for the standard ability block
        abilities_match = re.search(r'STR\s+(\d+)\s+DEX\s+(\d+)\s+CON\s+(\d+)\s+INT\s+(\d+)\s+WIS\s+(\d+)\s+CHA\s+(\d+)', text)
        if abilities_match:
            try:
                return MonsterAbilityScores(
                    strength=int(abilities_match.group(1)),
                    dexterity=int(abilities_match.group(2)),
                    constitution=int(abilities_match.group(3)),
                    intelligence=int(abilities_match.group(4)),
                    wisdom=int(abilities_match.group(5)),
                    charisma=int(abilities_match.group(6))
                )
            except (ValueError, IndexError):
                pass

        # Method 2: Look for individual ability scores
        abilities = {}
        for ability in ['STR', 'DEX', 'CON', 'INT', 'WIS', 'CHA']:
            match = re.search(f'{ability}\\s+(\\d+)', text)
            if match:
                try:
                    abilities[ability.lower()] = int(match.group(1))
                except ValueError:
                    abilities[ability.lower()] = 10

        # If we found at least some abilities, use them with defaults for missing ones
        if abilities:
            return MonsterAbilityScores(
                strength=abilities.get('str', 10),
                dexterity=abilities.get('dex', 10),
                constitution=abilities.get('con', 10),
                intelligence=abilities.get('int', 10),
                wisdom=abilities.get('wis', 10),
                charisma=abilities.get('cha', 10)
            )

        return MonsterAbilityScores(10, 10, 10, 10, 10, 10)  # Default abilities

    def _extract_attacks(self, text: str) -> List[MonsterAttack]:
        """Extract attacks with better pattern matching"""
        attacks = []

        # Pattern for melee/ranged attacks
        attack_patterns = [
            r'([A-Za-z\s]+)\s+[+–-](\d+)\s+([^,]+?)\s*,\s*([^.•]+)',
            r'([A-Za-z\s]+) Attack[:\•]\s*[+–-](\d+)[^,]*?([^,]+?)\s*,\s*([^.•]+)',
            r'•\s*([A-Za-z\s]+)[^+]+[+–-](\d+)[^,]*?([^,]+?)\s*,\s*([^.•]+)'
        ]

        for pattern in attack_patterns:
            matches = re.findall(pattern, text, re.IGNORECASE)
            for match in matches:
                try:
                    attack = MonsterAttack(
                        name=match[0].strip(),
                        attack_bonus=int(match[1]),
                        damage_dice=match[2].strip(),
                        damage_type=match[3].strip()
                    )
                    attacks.append(attack)
                except (ValueError, IndexError):
                    continue

        return attacks

    def _extract_attacks(self, text: str) -> List[MonsterAttack]:
        """Extract attacks from text"""
        attacks = []
        # Look for attack patterns
        attack_matches = re.findall(r'([A-Za-z\s]+)\s+[+–-](\d+)\s+([^,]+),\s*([^.•]+)', text)
        for match in attack_matches:
            attacks.append(MonsterAttack(
                name=match[0].strip(),
                attack_bonus=int(match[1]),
                damage_dice=match[2].strip(),
                damage_type=match[3].strip()
            ))
        return attacks

    def _extract_special_abilities(self, text: str) -> List[str]:
        """Extract special abilities"""
        abilities = []
        # Look for ability descriptions
        ability_matches = re.findall(r'([A-Z][^.•]+?)\.\s*([^•]+)', text)
        for match in ability_matches:
            abilities.append(f"{match[0]}: {match[1]}")
        return abilities

    def _extract_description(self, text: str) -> str:
        """Extract descriptive text"""
        lines = text.split('\n')
        description_lines = []
        for line in lines:
            if (line.strip() and not line.isupper() and
                not any(keyword in line for keyword in ['Armor Class', 'Hit Points', 'Speed', 'STR', 'DEX'])):
                description_lines.append(line.strip())
        return ' '.join(description_lines[:3])

    def _extract_environment(self, text: str) -> List[str]:
        """Extract environment info"""
        env_match = re.search(r'Environment[:\s]*([^\n]+)', text, re.IGNORECASE)
        if env_match:
            return [e.strip() for e in env_match.group(1).split(',')]
        return ["Any"]

# Integrated Pipeline
class MonsterPDFToChromaPipeline:
    """Complete pipeline: PDF -> Parse -> Chunk -> ChromaDB"""

    def __init__(self, db_path: str = "./chromadb"):
        self.pdf_parser = MonsterManualPDFParser()
        self.chunk_creator = MonsterChunkCreator()
        self.chroma_manager = MonsterChromaManager(db_path)

    def process_pdf_to_chromadb(self, pdf_path: str) -> int:
        """Process PDF and add monsters to ChromaDB"""
        print("🚀 Starting PDF to ChromaDB pipeline...")

        # Step 1: Extract PDF text
        text = self.pdf_parser.extract_pdf_text(pdf_path)
        if not text:
            print("❌ Failed to extract PDF text")
            return 0

        # Step 2: Split into monster blocks
        monster_blocks = self.pdf_parser.split_into_monster_blocks(text)

        # Step 3: Parse each block
        monsters = []
        successful_parses = 0

        for i, block in enumerate(monster_blocks):
            monster = self.pdf_parser.parse_monster_block(block)
            if monster:
                monsters.append(monster)
                successful_parses += 1
                if successful_parses % 10 == 0:
                    print(f"✅ Parsed {successful_parses} monsters...")

        print(f"📊 Successfully parsed {successful_parses}/{len(monster_blocks)} monsters")

        # Step 4: Create chunks and add to ChromaDB
        total_chunks = 0
        for monster in monsters:
            chunks = self.chunk_creator.create_monster_chunks(monster)
            self.chroma_manager.add_monster_chunks(chunks)
            total_chunks += len(chunks)

        print(f"🎉 Pipeline complete! Added {total_chunks} chunks from {len(monsters)} monsters")
        return total_chunks

# Usage Example
def main():
    # Initialize the pipeline
    pipeline = MonsterPDFToChromaPipeline()

    # Process your Monster Manual PDF
    pdf_path = "Dungeons and Dragons - Monster Manual (Skip Williams, Jonathan Tweet, Monte Cook) (Z-Library).pdf"  # Update this path!

    # Run the complete pipeline
    total_chunks = pipeline.process_pdf_to_chromadb(pdf_path)

    if total_chunks > 0:
        # Test that it worked
        results = pipeline.chroma_manager.search_monsters("dragon fire breath", n_results=3)
        print(f"🔍 Test search found {len(results['documents'][0])} results")

        # Show collection stats
        stats = pipeline.chroma_manager.get_collection_stats()
        print(f"📊 Final collection: {stats['total_documents']} documents")

if __name__ == "__main__":
    main()

🚀 Starting PDF to ChromaDB pipeline...
📖 Extracting text from Dungeons and Dragons - Monster Manual (Skip Williams, Jonathan Tweet, Monte Cook) (Z-Library).pdf...
Processing 334 pages...
Processed 50/334 pages...
Processed 100/334 pages...
Processed 150/334 pages...
Processed 200/334 pages...
Processed 250/334 pages...
Processed 300/334 pages...
✅ Extraction complete! 1,664,059 characters
📦 Found 480 potential monster blocks
✅ Parsed 10 monsters...
✅ Parsed 20 monsters...
✅ Parsed 30 monsters...
✅ Parsed 40 monsters...
✅ Parsed 50 monsters...
✅ Parsed 60 monsters...
✅ Parsed 70 monsters...
✅ Parsed 80 monsters...
✅ Parsed 90 monsters...
✅ Parsed 100 monsters...
✅ Parsed 110 monsters...
✅ Parsed 120 monsters...
✅ Parsed 130 monsters...
✅ Parsed 140 monsters...
✅ Parsed 150 monsters...
✅ Parsed 160 monsters...
✅ Parsed 170 monsters...
✅ Parsed 180 monsters...
✅ Parsed 190 monsters...
✅ Parsed 200 monsters...
✅ Parsed 210 monsters...
✅ Parsed 220 monsters...
✅ Parsed 230 monsters...
✅ Par

In [46]:
import chromadb

# Connect to your ChromaDB
client = chromadb.PersistentClient(path="./chromadb")
collection = client.get_collection("monsters")

# Query for goblin
results = collection.query(
    query_texts=["goblin"],
    n_results=10
)

print("🔍 Goblin Search Results:")
print("=" * 50)

if results["documents"] and results["documents"][0]:
    for i, (doc, meta, doc_id) in enumerate(zip(results["documents"][0], results["metadatas"][0], results["ids"][0])):
        print(f"\n📄 Result {i+1}:")
        print(f"   ID: {doc_id}")
        print(f"   Monster: {meta.get('monster_name', 'Unknown')}")
        print(f"   Chunk Type: {meta.get('chunk_type', 'Unknown')}")
        print(f"   CR: {meta.get('challenge_rating', 'Unknown')}")
        print(f"   Type: {meta.get('creature_type', 'Unknown')}")
        print(f"   Content snippet: {doc[:150]}...")
        print(f"   Metadata keys: {list(meta.keys())}")
else:
    print("❌ No goblin results found!")

🔍 Goblin Search Results:

📄 Result 1:
   ID: hobgoblin_society_combat_1
   Monster: Hobgoblin Society
   Chunk Type: combat
   CR: 1
   Type: humanoid
   Content snippet: **Hobgoblin Society** (combat)\n**Hobgoblin Society - Combat**\nCR 1, AC 10, HP 1\n**Attacks:**\n- **based and includes a:** +2 to hit, racial bonus.
...
   Metadata keys: ['monster_name', 'damage_resistances', 'challenge_rating', 'senses', 'name', 'environment', 'size', 'tags', 'languages', 'chunk_type', 'damage_immunities', 'condition_immunities', 'creature_type', 'alignment']

📄 Result 2:
   ID: area_10:_slave_quarters_(el_1)_traits_4
   Monster: Area 10: Slave Quarters (El 1)
   Chunk Type: traits
   CR: 6
   Type: humanoid
   Content snippet: **Area 10: Slave Quarters (El 1)** (traits)\n**Area 10: Slave Quarters (El 1) - Traits**\nCR 6\n**Traits:**\n- AREA 10: SLAVE QUARTERS (EL 1)
the entr...
   Metadata keys: ['tags', 'chunk_type', 'languages', 'monster_name', 'condition_immunities', 'size', 'senses', 'alignmen